In [2]:
getwd()

NameError: name 'getwd' is not defined

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("비어플/F_merged (1).csv")
df

,date,OilPrice,RealInterestRate,CPI,DollarIndex,VIX,IndustryProduction,CPE,OilInventories,OPECProduction,OilProduction,TermSpread,TreasuryYield,FedFundsRate
0,2006-01-01,NaN,2.064842,199.3,NaN,NaN,98.1999,NaN,NaN,33237.829,NaN,NaN,NaN,4.29
1,2006-01-02,NaN,NaN,NaN,101.4155,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2006-01-03,63.11,NaN,NaN,100.7558,11.14,NaN,NaN,NaN,NaN,NaN,0.03,4.37,NaN
3,2006-01-04,63.41,NaN,NaN,100.2288,11.37,NaN,NaN,NaN,NaN,NaN,0.05,4.36,NaN
4,2006-01-05,62.81,NaN,NaN,100.2992,11.31,NaN,NaN,NaN,NaN,NaN,0.04,4.36,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5339,2026-03-12,95.61,NaN,NaN,119.8227,27.29,NaN,NaN,NaN,NaN,NaN,0.51,4.27,NaN
5340,2026-03-13,98.48,NaN,NaN,120.5518,27.19,NaN,NaN,449259.0,NaN,13668.0,0.55,4.28,NaN
5341,2026-03-16,93.39,NaN,NaN,NaN,23.51,NaN,NaN,NaN,NaN,NaN,0.55,4.23,NaN
5342,2026-03-17,NaN,NaN,NaN,NaN,22.37,NaN,NaN,NaN,NaN,NaN,0.52,4.20,NaN


In [4]:
# 1. 날짜형 변환 + 정렬
# =========================
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print("원본 데이터 크기:", df.shape)
print("\n컬럼 목록:")
print(df.columns.tolist())

print("\n결측치 개수:")
print(df.isna().sum().sort_values(ascending=False))

# =========================
# 2. 변수 그룹 정의
# =========================
daily_market_vars = [
    "OilPrice", "DollarIndex", "VIX", "TermSpread", "TreasuryYield"
]

weekly_vars = [
    "OilInventories", "OilProduction"
]

monthly_vars = [
    "RealInterestRate", "CPI", "IndustryProduction",
    "CPE", "OPECProduction", "FedFundsRate"
]

# =========================
# 3. 거래일 캘린더 만들기
#    OilPrice가 존재하는 날짜를 거래일로 간주
# =========================
trading_calendar = (
    df.loc[df["OilPrice"].notna(), ["date"]]
      .drop_duplicates()
      .sort_values("date")
      .reset_index(drop=True)
)

print("\n거래일 수:", len(trading_calendar))

# =========================
# 4. 거래일 기준으로 merge
# =========================
df_trading = trading_calendar.merge(df, on="date", how="left")

# =========================
# 5. 주간 / 월간 변수 결측 처리
#    forward fill
# =========================
df_trading[weekly_vars + monthly_vars] = df_trading[weekly_vars + monthly_vars].ffill()

# =========================
# 6. 타깃 변수 생성
#    - OilReturn: 당일 수익률
#    - TargetReturn_t1: 다음 거래일 수익률
#    - TargetDirection_t1: 다음 거래일 상승 여부
# =========================
df_trading["OilReturn"] = df_trading["OilPrice"].pct_change()
df_trading["TargetReturn_t1"] = df_trading["OilReturn"].shift(-1)
df_trading["TargetDirection_t1"] = (df_trading["TargetReturn_t1"] > 0).astype(int)

# =========================
# 7. lag 변수 생성
# =========================
for col in daily_market_vars:
    df_trading[f"{col}_lag1"] = df_trading[col].shift(1)
    df_trading[f"{col}_lag5"] = df_trading[col].shift(5)

for col in weekly_vars + monthly_vars:
    df_trading[f"{col}_lag1"] = df_trading[col].shift(1)

# =========================
# 8. 최종 모델용 데이터셋 만들기
# =========================
feature_cols = (
    daily_market_vars
    + weekly_vars
    + monthly_vars
    + [f"{col}_lag1" for col in daily_market_vars + weekly_vars + monthly_vars]
    + [f"{col}_lag5" for col in daily_market_vars]
)

model_df = df_trading[["date", "TargetReturn_t1", "TargetDirection_t1"] + feature_cols].copy()
model_df = model_df.dropna().reset_index(drop=True)

# =========================
# 9. 결과 확인
# =========================
print("\n전처리 후 데이터 크기:", model_df.shape)
print("\n상위 5개 행:")
print(model_df.head())

print("\n최종 결측치 개수:")
print(model_df.isna().sum().sort_values(ascending=False).head(20))

# =========================
# 10. 저장
# =========================
df.to_csv("model_data_ffill_dropna.csv", index=False)
print("\n저장 완료: model_data_ffill_dropna.csv")

원본 데이터 크기: (5344, 14)

컬럼 목록:
['date', 'OilPrice', 'RealInterestRate', 'CPI', 'DollarIndex', 'VIX', 'IndustryProduction', 'CPE', 'OilInventories', 'OPECProduction', 'OilProduction', 'TermSpread', 'TreasuryYield', 'FedFundsRate']

결측치 개수:
CPE                   5115
OPECProduction        5105
CPI                   5103
IndustryProduction    5102
FedFundsRate          5102
RealInterestRate      5101
OilInventories        4290
OilProduction         4290
TreasuryYield          290
TermSpread             289
DollarIndex            282
OilPrice               275
VIX                    232
date                     0
dtype: int64

거래일 수: 5069

전처리 후 데이터 크기: (4618, 34)

상위 5개 행:
        date  TargetReturn_t1  TargetDirection_t1  OilPrice  DollarIndex  \
0 2007-02-02        -0.005423                   0     59.01      97.9743   
1 2007-02-05         0.003749                   1     58.69      97.8700   
2 2007-02-06        -0.019691                   0     58.91      97.7357   
3 2007-02-07      